# Multimodal Emotion Detection — Dual-Encoder (`FEEL-IT` + `ViT-224`) vs. Single Textual Encoder

**Task:** Multi-Label Emotion Classification (10 labels: Plutchik's 8 basic emotions + *Amore* + *Neutro*)  
- **Text Encoder**: Locally trained `FEEL-IT` checkpoints (`./checkpoints/emotion_textual_no_corr` for No Correlations; `./checkpoints/emotion_textual_corr` for With Correlations).
- **Vision Encoder**: [`google/vit-base-patch16-224`](https://huggingface.co/google/vit-base-patch16-224) (768-d `[CLS]` patch representation).
- **Classifier Head**: `ClassifierChain(SGDClassifier(loss='squared_hinge', penalty='l2', n_jobs=-1, random_state=42, class_weight='balanced'))` trained on the 1536-d joint `[CLS_text || CLS_img]` representation (Dual-Encoder) vs. the 768-d `[CLS_text]` representation (Text-Only baseline), matching `multimodal_separated_encoder_models.ipynb`.


In [8]:
# Install CUDA-enabled PyTorch (supports GTX 1080 Ti / Pascal via CUDA 12.1) and all requirements
%pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
%pip install -q -r ../requirements.txt


Note: you may need to restart the kernel to use updated packages.
^C
Note: you may need to restart the kernel to use updated packages.


## 1. Environment Setup & Paths


In [ ]:
import os
os.environ['TQDM_NOTEBOOK'] = 'false'
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.multioutput import ClassifierChain
from transformers import AutoModel, AutoTokenizer, pipeline, set_seed
import tqdm.auto, tqdm.std
tqdm.auto.tqdm = tqdm.std.tqdm  # Force plain-text progress bars in VS Code

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

repo_root = Path.cwd().resolve()
for _ in range(5):
    if (repo_root / "datasets").exists():
        break
    repo_root = repo_root.parent
REPO_ROOT = repo_root
DATASETS_DIR = REPO_ROOT / "datasets" / "corpora"
IMAGES_DIR = DATASETS_DIR / "images"
CHECKPOINTS_DIR = Path("./checkpoints")
RESULTS_DIR = Path("./results")
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MULTIMODAL_TRAIN_CSV = DATASETS_DIR / "correctly_split" / "multimodal_train_seed42.csv"
MULTIMODAL_TEST_CSV = DATASETS_DIR / "correctly_split" / "multimodal_test_seed42.csv"

EMOTION_LABELS = [
    "Amore", "Anticipazione", "Disgusto", "Fiducia", "Gioia",
    "Neutro", "Paura", "Rabbia", "Sorpresa", "Tristezza"
]

LOCAL_CKPT_NO_CORR = str(CHECKPOINTS_DIR / "emotion_textual_no_corr")
LOCAL_CKPT_WITH_CORR = str(CHECKPOINTS_DIR / "emotion_textual_corr")

VIT_MODEL_ID = "google/vit-base-patch16-224" 


Using device: cuda:0


## 2. Helper Functions for Feature Extraction & Evaluation


In [ ]:
def create_type_column(row):
    """Matches semiotic relation subset assignment (1=Additiva, 2=Parallela, 3=Divergente)."""
    if row["Additiva"] == 1:
        return 1
    elif row["Parallela"] == 1:
        return 2
    elif row["Divergente"] == 1:
        return 3
    else:
        return 0


def extract_text_cls_features(
    texts: list,
    model_path: str,
    device: str,
    max_length: int = 128,
    batch_size: int = 32,
) -> torch.Tensor:
    """Extracts 768-d [CLS] embeddings from the last hidden state of the local text encoder checkpoint."""
    if not os.path.exists(os.path.join(model_path, "config.json")):
        raise FileNotFoundError(
            f"Checkpoint not found at {model_path}. Please run train_textual_emotion.ipynb first."
        )
    print(f"Loading local checkpoint: {model_path}")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    txt_model = AutoModel.from_pretrained(model_path).to(device)
    txt_model.eval()

    all_cls = []
    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch_texts = [str(t) for t in texts[start : start + batch_size]]
            tokenized = tokenizer(
                batch_texts,
                padding="max_length",
                truncation=True,
                max_length=max_length,
                return_token_type_ids=True,
                return_tensors="pt",
            ).to(device)
            outputs = txt_model(
                input_ids=tokenized.input_ids,
                attention_mask=tokenized.attention_mask,
                output_hidden_states=True,
            )
            cls_emb = outputs.hidden_states[-1][:, 0, :].cpu()
            all_cls.append(cls_emb)

    return torch.cat(all_cls, dim=0)


def load_rgb_images(img_paths: pd.Series, images_dir: Path) -> list:
    images = []
    for rel_path in img_paths:
        full_path = images_dir / str(rel_path)
        img = Image.open(full_path)
        if img.mode != "RGB":
            img = img.convert("RGB")
        images.append(img)
    return images


def extract_vit_cls_features(images: list, pipe_images, batch_size: int = 32) -> torch.Tensor:
    """Extracts 768-d [CLS] visual features from google/vit-base-patch16-224."""
    all_img_cls = []
    for start in range(0, len(images), batch_size):
        batch_imgs = images[start : start + batch_size]
        imgs_out = pipe_images(batch_imgs, return_tensors=True, framework="pt")
        batch_cls = torch.stack([t[0, 0, :].cpu() for t in imgs_out], dim=0)
        all_img_cls.append(batch_cls)
    return torch.cat(all_img_cls, dim=0)


def evaluate_multimodal_regime(
    y_true_test: np.ndarray,
    y_pred_test: np.ndarray,
    test_types: np.ndarray,
    target_names: list,
) -> dict:
    report_normal = classification_report(
        y_true_test, y_pred_test, target_names=target_names, output_dict=True, zero_division=0
    )
    mask_add = test_types == 1
    mask_par = test_types == 2
    mask_div = (test_types == 3) | (test_types == 0)

    report_add = classification_report(
        y_true_test[mask_add], y_pred_test[mask_add], target_names=target_names, output_dict=True, zero_division=0
    )
    report_par = classification_report(
        y_true_test[mask_par], y_pred_test[mask_par], target_names=target_names, output_dict=True, zero_division=0
    )
    report_div = classification_report(
        y_true_test[mask_div], y_pred_test[mask_div], target_names=target_names, output_dict=True, zero_division=0
    )

    return {
        "normal": report_normal,
        "add": report_add,
        "par": report_par,
        "div": report_div,
    }


## 3. Load Multimodal Datasets (80/20 Split) & Extract Shared `ViT-224` Image Features


In [ ]:
def build_multimodal_corr_text(df_in, emotions=False, target=False, irony=False, hate=False, relations=False):
    """
    Exact correlation prefix generator from italian_correlations.py.
    Verified 100% identical to emotions_multimodal_corr_ita.csv,
    irony_multimodal_corr_ita.csv, and relation_multimodal_corr_ita.csv.
    """
    emotion_cols = [
        "Amore", "Anticipazione", "Disgusto", "Fiducia", "Gioia",
        "Neutro", "Paura", "Rabbia", "Sorpresa", "Tristezza"
    ]
    out_texts = []
    for _, row in df_in.iterrows():
        em, tg, ih, rel = [], [], [], []
        if emotions:
            for c in emotion_cols:
                if int(row.get(c, 0)) == 1:
                    em.append(c.lower())
        if target:
            if int(row.get("Contenuto", 0)) == 1:
                tg.append("contenuto")
            if int(row.get("Contenitore", 0)) == 1:
                tg.append("contenitore")
        if irony:
            if int(row.get("Ironia", 0)) == 1:
                ih.append("ironia")
            if int(row.get("Sarcasmo", 0)) == 1:
                ih.append("sarcasmo")
        if hate:
            if int(row.get("Offensivita", 0)) == 1:
                ih.append("offensivita")
        if relations:
            if int(row.get("Additiva", 0)) == 1:
                rel.append("additiva")
            if int(row.get("Divergente", 0)) == 1:
                rel.append("divergente")
            if int(row.get("Parallela", 0)) == 1:
                rel.append("parallela")
        text = str(row["text"])
        if tg:
            text = "riguarda {} ".format(" e ".join(tg)) + text
        if ih:
            text = "contiene {} ".format(" e ".join(ih)) + text
        if rel:
            text = "ha una relazione {} ".format(" e ".join(rel)) + text
        if em:
            text = "esprime {} ".format(" e ".join(em)) + text
        text = "il seguente testo " + text
        out_texts.append(text)
    return out_texts

# Load pre-split multimodal train (516) and test (129) sets from datasets/corpora/correctly_split/
train_full_nc = pd.read_csv(MULTIMODAL_TRAIN_CSV)
test_nc = pd.read_csv(MULTIMODAL_TEST_CSV)

train_full_nc["type"] = train_full_nc.apply(create_type_column, axis=1)
test_nc["type"] = test_nc.apply(create_type_column, axis=1)

train_full_wc = train_full_nc.copy()
test_wc = test_nc.copy()
train_full_wc["text"] = build_multimodal_corr_text(train_full_wc, target=True, irony=True, hate=True, relations=True)
test_wc["text"] = build_multimodal_corr_text(test_wc, target=True, irony=True, hate=True, relations=True)

train_full_nc = train_full_nc.reset_index(drop=True)
test_nc = test_nc.reset_index(drop=True)
train_full_wc = train_full_wc.reset_index(drop=True)
test_wc = test_wc.reset_index(drop=True)

print(f"Multimodal Split -> Train: {len(train_full_nc)} | Test: {len(test_nc)}")

y_train = train_full_nc[EMOTION_LABELS].values.astype(int)
y_test = test_nc[EMOTION_LABELS].values.astype(int)
test_types = test_nc["type"].values

print(f"Loading Vision Encoder: {VIT_MODEL_ID} ...")
pipe_images = pipeline(task="image-feature-extraction", model=VIT_MODEL_ID, device=device)

print("Extracting ViT-224 features for training and test images...")
train_images = load_rgb_images(train_full_nc["img_path"], IMAGES_DIR)
test_images = load_rgb_images(test_nc["img_path"], IMAGES_DIR)

h_img_train = extract_vit_cls_features(train_images, pipe_images)
h_img_test = extract_vit_cls_features(test_images, pipe_images)
print(f"ViT Train Features shape: {h_img_train.shape} | ViT Test Features shape: {h_img_test.shape}")


Multimodal Split -> Train: 516 | Test: 129
Loading Vision Encoder: google/vit-base-patch16-224 ...


Some weights of ViTModel were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.
Device set to use cuda:0


Extracting ViT-224 features for training and test images...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


ViT Train Features shape: torch.Size([516, 768]) | ViT Test Features shape: torch.Size([129, 768])


## 4. Regime 1 — Without Correlations: Single Textual Encoder vs. Dual-Encoder (`FEEL-IT` + `ViT-224`)


In [ ]:
h_txt_train_nc = extract_text_cls_features(train_full_nc["text"].tolist(), LOCAL_CKPT_NO_CORR, device)
h_txt_test_nc = extract_text_cls_features(test_nc["text"].tolist(), LOCAL_CKPT_NO_CORR, device)

# 4A. Blind Text-Only Baseline on Multimodal Dataset (Without Correlations)
clf_txt_nc = ClassifierChain(
    SGDClassifier(loss="squared_hinge", penalty="l2", n_jobs=-1, random_state=SEED, class_weight="balanced")
)
clf_txt_nc.fit(h_txt_train_nc.numpy(), y_train)
y_pred_txt_nc = clf_txt_nc.predict(h_txt_test_nc.numpy()).astype(int)
res_txt_nc = evaluate_multimodal_regime(y_test, y_pred_txt_nc, test_types, EMOTION_LABELS)

print("=== [No Correlations] Blind Text-Only Baseline (129 Test Posts) ===")
print(classification_report(y_test, y_pred_txt_nc, target_names=EMOTION_LABELS, digits=4, zero_division=0))

# 4B. Dual-Encoder Text + Image (Without Correlations)
h_joint_train_nc = torch.cat((h_txt_train_nc, h_img_train), dim=1)
h_joint_test_nc = torch.cat((h_txt_test_nc, h_img_test), dim=1)
print(f"Joint Multimodal Embedding shape: {h_joint_train_nc.shape}")

clf_mm_nc = ClassifierChain(
    SGDClassifier(loss="squared_hinge", penalty="l2", n_jobs=-1, random_state=SEED, class_weight="balanced")
)
clf_mm_nc.fit(h_joint_train_nc.numpy(), y_train)
y_pred_mm_nc = clf_mm_nc.predict(h_joint_test_nc.numpy()).astype(int)
res_mm_nc = evaluate_multimodal_regime(y_test, y_pred_mm_nc, test_types, EMOTION_LABELS)

print("=== [No Correlations] Dual-Encoder Text + Image (129 Test Posts) ===")
print(classification_report(y_test, y_pred_mm_nc, target_names=EMOTION_LABELS, digits=4, zero_division=0))


Loading local checkpoint: d:\vsc_projects\MultiEMit\MultiEMit\code\bert\checkpoints\seq_emotion_step2_no_corr


Some weights of CamembertModel were not initialized from the model checkpoint at d:\vsc_projects\MultiEMit\MultiEMit\code\bert\checkpoints\seq_emotion_step2_no_corr and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading local checkpoint: d:\vsc_projects\MultiEMit\MultiEMit\code\bert\checkpoints\seq_emotion_step2_no_corr


Some weights of CamembertModel were not initialized from the model checkpoint at d:\vsc_projects\MultiEMit\MultiEMit\code\bert\checkpoints\seq_emotion_step2_no_corr and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


=== [No Correlations] Blind Text-Only Baseline (129 Test Posts) ===
               precision    recall  f1-score   support

        Amore     0.1930    1.0000    0.3235        11
Anticipazione     0.4500    0.9000    0.6000        10
     Disgusto     0.4483    0.6842    0.5417        19
      Fiducia     0.5077    0.7857    0.6168        42
        Gioia     0.7333    0.3793    0.5000        29
       Neutro     0.2500    0.0588    0.0952        17
        Paura     0.0000    0.0000    0.0000         1
       Rabbia     0.1250    0.1667    0.1429         6
     Sorpresa     0.0000    0.0000    0.0000        16
    Tristezza     0.0444    1.0000    0.0851         2

    micro avg     0.2978    0.5294    0.3812       153
    macro avg     0.2752    0.4975    0.2905       153
 weighted avg     0.4106    0.5294    0.4111       153
  samples avg     0.2997    0.5194    0.3632       153

Joint Multimodal Embedding shape: torch.Size([516, 1536])
=== [No Correlations] Dual-Encoder Text + Imag

## 5. Regime 2 — With Correlations: Single Textual Encoder vs. Dual-Encoder (`FEEL-IT` + `ViT-224`)


In [ ]:
h_txt_train_wc = extract_text_cls_features(train_full_wc["text"].tolist(), LOCAL_CKPT_WITH_CORR, device)
h_txt_test_wc = extract_text_cls_features(test_wc["text"].tolist(), LOCAL_CKPT_WITH_CORR, device)

# 5A. Blind Text-Only Baseline on Multimodal Dataset (With Correlations)
clf_txt_wc = ClassifierChain(
    SGDClassifier(loss="squared_hinge", penalty="l2", n_jobs=-1, random_state=SEED, class_weight="balanced")
)
clf_txt_wc.fit(h_txt_train_wc.numpy(), y_train)
y_pred_txt_wc = clf_txt_wc.predict(h_txt_test_wc.numpy()).astype(int)
res_txt_wc = evaluate_multimodal_regime(y_test, y_pred_txt_wc, test_types, EMOTION_LABELS)

print("=== [With Correlations] Blind Text-Only Baseline (129 Test Posts) ===")
print(classification_report(y_test, y_pred_txt_wc, target_names=EMOTION_LABELS, digits=4, zero_division=0))

# 5B. Dual-Encoder Text + Image (With Correlations)
h_joint_train_wc = torch.cat((h_txt_train_wc, h_img_train), dim=1)
h_joint_test_wc = torch.cat((h_txt_test_wc, h_img_test), dim=1)

clf_mm_wc = ClassifierChain(
    SGDClassifier(loss="squared_hinge", penalty="l2", n_jobs=-1, random_state=SEED, class_weight="balanced")
)
clf_mm_wc.fit(h_joint_train_wc.numpy(), y_train)
y_pred_mm_wc = clf_mm_wc.predict(h_joint_test_wc.numpy()).astype(int)
res_mm_wc = evaluate_multimodal_regime(y_test, y_pred_mm_wc, test_types, EMOTION_LABELS)

print("=== [With Correlations] Dual-Encoder Text + Image (129 Test Posts) ===")
print(classification_report(y_test, y_pred_mm_wc, target_names=EMOTION_LABELS, digits=4, zero_division=0))


Loading local checkpoint: d:\vsc_projects\MultiEMit\MultiEMit\code\bert\checkpoints\seq_emotion_step3_corr


Some weights of CamembertModel were not initialized from the model checkpoint at d:\vsc_projects\MultiEMit\MultiEMit\code\bert\checkpoints\seq_emotion_step3_corr and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of CamembertModel were not initialized from the model checkpoint at d:\vsc_projects\MultiEMit\MultiEMit\code\bert\checkpoints\seq_emotion_step3_corr and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading local checkpoint: d:\vsc_projects\MultiEMit\MultiEMit\code\bert\checkpoints\seq_emotion_step3_corr
=== [With Correlations] Blind Text-Only Baseline (129 Test Posts) ===
               precision    recall  f1-score   support

        Amore     0.3750    0.2727    0.3158        11
Anticipazione     0.2963    0.8000    0.4324        10
     Disgusto     0.4286    0.7895    0.5556        19
      Fiducia     0.4783    0.5238    0.5000        42
        Gioia     0.5455    0.2069    0.3000        29
       Neutro     0.7143    0.2941    0.4167        17
        Paura     0.0000    0.0000    0.0000         1
       Rabbia     0.2857    0.3333    0.3077         6
     Sorpresa     0.1739    0.5000    0.2581        16
    Tristezza     0.0000    0.0000    0.0000         2

    micro avg     0.3301    0.4510    0.3812       153
    macro avg     0.3297    0.3720    0.3086       153
 weighted avg     0.4430    0.4510    0.3994       153
  samples avg     0.3669    0.4651    0.3886       

## 6. Final Comparison Tables & Per-Label Classification Reports


In [ ]:
from sklearn.metrics import classification_report, f1_score, accuracy_score

os.makedirs(RESULTS_DIR, exist_ok=True)

rep_txt_nc = classification_report(y_test, y_pred_txt_nc, target_names=EMOTION_LABELS, digits=4, zero_division=0)
rep_mm_nc = classification_report(y_test, y_pred_mm_nc, target_names=EMOTION_LABELS, digits=4, zero_division=0)
rep_txt_wc = classification_report(y_test, y_pred_txt_wc, target_names=EMOTION_LABELS, digits=4, zero_division=0)
rep_mm_wc = classification_report(y_test, y_pred_mm_wc, target_names=EMOTION_LABELS, digits=4, zero_division=0)

print('=== [1/4] Text-Only (FEEL-IT Sequential) — WITHOUT Correlations ===')
print(rep_txt_nc)

print('\n=== [2/4] Multimodal Dual-Encoder (FEEL-IT Sequential + ViT-224) — WITHOUT Correlations ===')
print(rep_mm_nc)

print('\n=== [3/4] Text-Only (FEEL-IT Sequential) — WITH Correlations ===')
print(rep_txt_wc)

print('\n=== [4/4] Multimodal Dual-Encoder (FEEL-IT Sequential + ViT-224) — WITH Correlations ===')
print(rep_mm_wc)

def summarize_regime(modality, name, y_true, y_pred):
    return {
        'Task': 'Emotion Detection (10-Label Multi-Label)',
        'Modality': modality,
        'Notebook': 'multimodal_dual_encoder_emotion.ipynb',
        'Configuration': name,
        'Micro F1': f1_score(y_true, y_pred, average='micro', zero_division=0),
        'Macro F1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'Weighted F1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'Accuracy / Subset Acc': accuracy_score(y_true, y_pred),
    }

rows = [
    summarize_regime('Multimodal Split (Text-Only SGD)', 'Text-Only (FEEL-IT Sequential) — No Correlations', y_test, y_pred_txt_nc),
    summarize_regime('Multimodal Dual-Encoder (Text + ViT-224)', 'Multimodal Dual-Encoder (FEEL-IT Sequential + ViT-224) — No Correlations', y_test, y_pred_mm_nc),
    summarize_regime('Multimodal Split (Text-Only SGD)', 'Text-Only (FEEL-IT Sequential) — With Correlations', y_test, y_pred_txt_wc),
    summarize_regime('Multimodal Dual-Encoder (Text + ViT-224)', 'Multimodal Dual-Encoder (FEEL-IT Sequential + ViT-224) — With Correlations', y_test, y_pred_mm_wc),
]

report_blocks = [
    ('=== [1] Text-Only (FEEL-IT Sequential) — WITHOUT Correlations ===', rep_txt_nc),
    ('=== [2] Multimodal Dual-Encoder (FEEL-IT Sequential + ViT-224) — WITHOUT Correlations ===', rep_mm_nc),
    ('=== [3] Text-Only (FEEL-IT Sequential) — WITH Correlations ===', rep_txt_wc),
    ('=== [4] Multimodal Dual-Encoder (FEEL-IT Sequential + ViT-224) — WITH Correlations ===', rep_mm_wc),
]

# Also evaluate Concatenated checkpoints if present on disk
concat_nc_dir = CHECKPOINTS_DIR / 'concat_emotion_no_corr'
concat_wc_dir = CHECKPOINTS_DIR / 'concat_emotion_corr'
if (concat_nc_dir / 'config.json').exists() and (concat_wc_dir / 'config.json').exists():
    print('\nEvaluating Concatenated FEEL-IT checkpoints on Multimodal Split...')
    h_txt_tr_cnc = extract_text_cls_features(train_full_nc['text'].tolist(), str(concat_nc_dir), device)
    h_txt_te_cnc = extract_text_cls_features(test_nc['text'].tolist(), str(concat_nc_dir), device)

    clf_t_cnc = ClassifierChain(SGDClassifier(loss='squared_hinge', penalty='l2', n_jobs=-1, random_state=SEED, class_weight='balanced'), order='random', random_state=SEED)
    clf_t_cnc.fit(h_txt_tr_cnc, y_train)
    yp_t_cnc = clf_t_cnc.predict(h_txt_te_cnc).astype(int)

    clf_m_cnc = ClassifierChain(SGDClassifier(loss='squared_hinge', penalty='l2', n_jobs=-1, random_state=SEED, class_weight='balanced'), order='random', random_state=SEED)
    clf_m_cnc.fit(np.concatenate([h_txt_tr_cnc, h_img_train], axis=1), y_train)
    yp_m_cnc = clf_m_cnc.predict(np.concatenate([h_txt_te_cnc, h_img_test], axis=1)).astype(int)

    h_txt_tr_cwc = extract_text_cls_features(train_full_wc['text'].tolist(), str(concat_wc_dir), device)
    h_txt_te_cwc = extract_text_cls_features(test_wc['text'].tolist(), str(concat_wc_dir), device)

    clf_t_cwc = ClassifierChain(SGDClassifier(loss='squared_hinge', penalty='l2', n_jobs=-1, random_state=SEED, class_weight='balanced'), order='random', random_state=SEED)
    clf_t_cwc.fit(h_txt_tr_cwc, y_train)
    yp_t_cwc = clf_t_cwc.predict(h_txt_te_cwc).astype(int)

    clf_m_cwc = ClassifierChain(SGDClassifier(loss='squared_hinge', penalty='l2', n_jobs=-1, random_state=SEED, class_weight='balanced'), order='random', random_state=SEED)
    clf_m_cwc.fit(np.concatenate([h_txt_tr_cwc, h_img_train], axis=1), y_train)
    yp_m_cwc = clf_m_cwc.predict(np.concatenate([h_txt_te_cwc, h_img_test], axis=1)).astype(int)

    rows.extend([
        summarize_regime('Multimodal Split (Text-Only SGD)', 'Text-Only (FEEL-IT Concatenated) — No Correlations', y_test, yp_t_cnc),
        summarize_regime('Multimodal Dual-Encoder (Text + ViT-224)', 'Multimodal Dual-Encoder (FEEL-IT Concatenated + ViT-224) — No Correlations', y_test, yp_m_cnc),
        summarize_regime('Multimodal Split (Text-Only SGD)', 'Text-Only (FEEL-IT Concatenated) — With Correlations', y_test, yp_t_cwc),
        summarize_regime('Multimodal Dual-Encoder (Text + ViT-224)', 'Multimodal Dual-Encoder (FEEL-IT Concatenated + ViT-224) — With Correlations', y_test, yp_m_cwc),
    ])
    report_blocks.extend([
        ('=== [5] Text-Only (FEEL-IT Concatenated) — WITHOUT Correlations ===', classification_report(y_test, yp_t_cnc, target_names=EMOTION_LABELS, digits=4, zero_division=0)),
        ('=== [6] Multimodal Dual-Encoder (FEEL-IT Concatenated + ViT-224) — WITHOUT Correlations ===', classification_report(y_test, yp_m_cnc, target_names=EMOTION_LABELS, digits=4, zero_division=0)),
        ('=== [7] Text-Only (FEEL-IT Concatenated) — WITH Correlations ===', classification_report(y_test, yp_t_cwc, target_names=EMOTION_LABELS, digits=4, zero_division=0)),
        ('=== [8] Multimodal Dual-Encoder (FEEL-IT Concatenated + ViT-224) — WITH Correlations ===', classification_report(y_test, yp_m_cwc, target_names=EMOTION_LABELS, digits=4, zero_division=0)),
    ])

results_table = pd.DataFrame(rows)
results_table.to_csv(RESULTS_DIR / 'multimodal_dual_encoder_emotion_summary.csv', index=False)
with open(RESULTS_DIR / 'multimodal_dual_encoder_emotion_reports.txt', 'w', encoding='utf-8') as f_rep:
    for title, rep in report_blocks:
        f_rep.write(title + '\n' + rep + '\n\n')

display(results_table.round(4))


=== [1/4] Text-Only (FEEL-IT) — WITHOUT Correlations ===
               precision    recall  f1-score   support

        Amore     0.1930    1.0000    0.3235        11
Anticipazione     0.4500    0.9000    0.6000        10
     Disgusto     0.4483    0.6842    0.5417        19
      Fiducia     0.5077    0.7857    0.6168        42
        Gioia     0.7333    0.3793    0.5000        29
       Neutro     0.2500    0.0588    0.0952        17
        Paura     0.0000    0.0000    0.0000         1
       Rabbia     0.1250    0.1667    0.1429         6
     Sorpresa     0.0000    0.0000    0.0000        16
    Tristezza     0.0444    1.0000    0.0851         2

    micro avg     0.2978    0.5294    0.3812       153
    macro avg     0.2752    0.4975    0.2905       153
 weighted avg     0.4106    0.5294    0.4111       153
  samples avg     0.2997    0.5194    0.3632       153


=== [2/4] Multimodal Dual-Encoder (FEEL-IT + ViT-224) — WITHOUT Correlations ===
               precision    recal

,Experiment,Micro F1,Macro F1,Weighted F1,Subset Accuracy
0,Text-Only (FEEL-IT) — No Correlations,0.3812,0.2905,0.4111,0.0698
1,Multimodal Dual-Encoder (FEEL-IT + ViT-224) — ...,0.4765,0.4191,0.4717,0.2326
2,Text-Only (FEEL-IT) — With Correlations,0.3812,0.3086,0.3994,0.2093
3,Multimodal Dual-Encoder (FEEL-IT + ViT-224) — ...,0.4859,0.4408,0.4848,0.2868
